# 5 · Collision cross sections

The electron–water collision tables used by the model (`XS/*.nc` in this repository, also bundled
inside the package so `cc.load_cross_sections()` works anywhere).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import collide_comet as cc
from collide_comet import plotting as ccp
from run_config import make_saver

save = make_saver("cross_sections", demo=False)   # these are real tables, not demo data

In [ ]:
xs = cc.load_cross_sections()                      # Coll_XS.nc
print("energies :", xs.energy.size, f"from {xs.energy[0]} to {xs.energy[-1]} eV")
for name, e_th in zip(cc.constants.PROCESS_NAMES, xs.e_th):
    print(f"  {name:22s} threshold {e_th:7.3f} eV")

In [ ]:
fig, axes = ccp.plot_cross_sections(xs)
save(fig, "collision_cross_sections");

Energy-loss cross sections (`inelasticXSplots.m`): threshold energy times cross section for excitations, mean energy lost for ionisation.

In [ ]:
fig, ax = ccp.plot_loss_rates(xs)
save(fig, "loss_rates");

## Effective cross section for producing cold electrons

The weighting used by `cold_electron_production` (`calculateColdeProduction.m`): for each energy bin
of a run, the summed cross section of all processes that leave an electron below `e_cold`.

In [ ]:
energy_edges = np.r_[np.arange(0, 10, 0.5), np.arange(10, 60, 5), np.arange(60, 401, 20)]
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for e_cold in [0.5, 1.0, 2.0]:
    xs_cold, _ = cc.physics.cold_electron_cross_section(energy_edges, xs, e_cold=e_cold)
    ax.stairs(xs_cold, energy_edges, label=f"E_cold = {e_cold} eV", linewidth=1.6)
ax.set_xlim(0, 70)
ax.set_yscale("log")
ax.set_xlabel("Electron energy [eV]")
ax.set_ylabel(r"Effective cross section [cm$^2$]")
ax.legend()
save(fig, "cold_electron_cross_section");

## The other bundled tables

In [ ]:
adv = cc.load_cross_sections("Adv_scat_Coll_XS.nc")    # adds angular scattering CDFs
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for i, energy in enumerate(adv.scat_energy):
    ax.plot(adv.scat_theta, adv.scat_cdf[i], label=f"{energy:g} eV")
ax.set_xlabel("Scattering angle [deg]")
ax.set_ylabel("Cumulative probability")
ax.legend(title="Electron energy", ncol=2, fontsize=8)
save(fig, "scattering_cdf");